# 06 · Energy resolution at Q_bb (0νββ)

**Purpose.** Plan and skeleton for the 0νββ energy resolution with fibers (and PMTs for comparison).

**Inputs.** nexus `20250205_Next100_bb_300ev_ELoff` (42 × 300 events, 10 bar, SECTOR_AREA) → detsim → hypathia.

**Outputs.** energy per event, resolution at 2458 keV.

**What is checked.** containment, lifetime correction per S2 slice, (x, y) correction from the light table (no Kr map needed: the simulation has no z-dependent geometry effect).

**Next step.** Write-up in the thesis.

In [ ]:
# Setup: run `source env.sh` before starting Jupyter, or pick the IC-3.8-2024-06-08 kernel.
import os, sys, glob
sys.path.insert(0, os.path.abspath(".."))
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import warnings; warnings.filterwarnings("ignore")
from icana import paths, checks, fits
print("ICDIR =", paths.ICDIR or "NOT SET (source env.sh)")

## Steps
1. `./submit.sh detsim fibers 20250205_Next100_bb_300ev_ELoff` (dry run), then add `--submit`. Same for `pmts`.
2. `./submit.sh hypathia fibers 20250205_Next100_bb_300ev_ELoff --submit` once detsim is done (and `pmts`).
3. Here: for each event, sum the S2 slices, correct each slice for attachment with exp(Δt/τ) using its time after S1, and for the (x, y) light collection with the map from `02_light_tables` (fibers) or the PMT table.
4. Select events fully contained (no energy near the walls or ends) with one S1.
5. Fit the peak at Q_bb and quote FWHM/E. Compare with the E^-1/2 extrapolation of the Kr result.

Open choices (decide before running): `el_gain` (365 as for Kr, or 1050), diffusion (IC defaults or Simón et al. 2018 at 10 bar), fiducial cuts.

In [ ]:
SAMPLE = "20250205_Next100_bb_300ev_ELoff"
for det in ["fibers", "pmts"]:
    for city in ["detsim", "hypathia"]:
        print(det, city, len(paths.new_files(SAMPLE, det, city)), "files")